# Méthodes bayésiennes : croire, observer, mettre à jour

Le raisonnement bayésien part d'une croyance (l'*a priori*), la confronte aux données (la *vraisemblance*) et en tire une croyance mise à jour (l'*a posteriori*). Ce TP le fait voir sur des cas courts, chacun avec ses propres données (environ 1 h, plus 35 min pour les exercices facultatifs) :

1. **Le taux de conversion d'une boutique en ligne** (15 min) : un a priori Bêta mis à jour au fil des visites ; maximum de vraisemblance, maximum a posteriori et moyenne a posteriori.
2. **Reconnaître un vin** (10 min) : un classifieur bayésien naïf de scikit-learn sur 178 vins.
3. **La pluie, l'arroseur et l'herbe mouillée** (15 min) : l'inférence dans le réseau bayésien du cours, par énumération.
4. **La météo cachée** (15 min) : le modèle de Markov caché du cours, décodé par l'algorithme de Viterbi.
5. **Facultatif — Une température et son incertitude** (15 min) : une régression par processus gaussien.
6. **Facultatif — Trois variantes d'une page** (20 min) : un bandit à trois bras, ε-glouton contre l'échantillonnage de Thompson.

Les exercices sont indépendants : chacun se fait sans les autres.

## Installation et imports

`hmmlearn`, la bibliothèque des modèles de Markov cachés, n'est pas installée sur Colab : la première cellule l'installe.

In [ ]:
%pip install -q hmmlearn

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM
from scipy import stats
from sklearn.datasets import load_wine
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

## 1. Le taux de conversion d'une boutique en ligne

Une boutique en ligne lance une nouvelle campagne et veut connaître son taux de conversion : la part des visites qui finissent par un achat. Avant toute visite, l'équipe marketing en a une idée : les campagnes passées tournaient autour de 20 %. C'est l'**a priori**, une loi Bêta de paramètres $\alpha = 2$ et $\beta = 8$, dont la moyenne vaut $\alpha / (\alpha + \beta) = 20$ %.

Puis les visites arrivent : chacune finit par un achat (1) ou non (0). Les données sont simulées : le vrai taux, que la boutique ne connaît pas, est de 12 %.

In [ ]:
alpha, beta = 2, 8  # a priori : autour de 20 % de conversion
achats = (np.random.default_rng(130).random(500) < 0.12).astype(int)
print(f"{len(achats)} visites, {achats.sum()} achats")
print("Les 10 premières :", achats[:10])

Après $n$ achats sur $N$ visites, l'a posteriori est encore une loi Bêta : les succès s'ajoutent à $\alpha$, les échecs à $\beta$, soit $\mathrm{Beta}(\alpha + n, \beta + N - n)$.

*Écrivez la fonction `a_posteriori(alpha, beta, n, N)`, qui renvoie cette loi (un objet `scipy.stats.beta`). Puis, après les 10 premières visites, affichez trois estimations du taux de conversion :*

- *le maximum de vraisemblance (MLE), qui ne regarde que les données : la fréquence observée $n / N$ ;*
- *le maximum a posteriori (MAP) : le taux où la densité a posteriori est la plus haute, cherché sur une grille de taux entre 0 et 1 ;*
- *la moyenne a posteriori, qui est aussi la probabilité que la prochaine visite finisse par un achat (la prédictive).*

*Que conclut le maximum de vraisemblance après 10 visites ? Que font l'a priori et les deux autres estimations ?*

**Documentation utile**

- [`scipy.stats.beta`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html)
- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)

In [ ]:
def a_posteriori(alpha, beta, n, N):
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

- `stats.beta(a, b)` crée la loi Bêta de paramètres `a` et `b` ; sa méthode `.pdf(x)` donne la densité en chaque taux du tableau `x`, et `.mean()` sa moyenne.
- Les 10 premières visites : `achats[:10]`. `n` est leur somme (le nombre d'achats), `N` leur nombre.
- MAP : `grille = np.linspace(0, 1, 1001)`, 1 001 taux de 0 à 1 ; `np.argmax(loi.pdf(grille))` donne la position du plus haut point de la densité, et `grille[...]` le taux correspondant.
- Valeurs attendues : 3 achats sur les 10 premières visites, donc un MLE de 30,0 %, un MAP de 22,2 % et une moyenne a posteriori de 25,0 %.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def a_posteriori(alpha, beta, n, N):
  return stats.beta(..., ...)


N = 10
n = achats[:N].sum()
loi = a_posteriori(alpha, beta, n, N)
grille = np.linspace(0, 1, 1001)
print(f"MLE : {...:.1%}")
print(f"MAP : {grille[np.argmax(...)]:.1%}")
print(f"Moyenne a posteriori : {...:.1%}")
```

### Solution

In [ ]:
def a_posteriori(alpha, beta, n, N):
  return stats.beta(alpha + n, beta + N - n)


N = 10
n = achats[:N].sum()
loi = a_posteriori(alpha, beta, n, N)
grille = np.linspace(0, 1, 1001)
print(f"MLE : {n / N:.1%}")
print(f"MAP : {grille[np.argmax(loi.pdf(grille))]:.1%}")
print(f"Moyenne a posteriori : {loi.mean():.1%}")

3 achats sur 10 visites : le maximum de vraisemblance conclut à 30 % de conversion, sur la foi de 10 visites seulement. L'a priori retient cette conclusion hâtive : le MAP (22,2 %) et la moyenne a posteriori (25 %) restent entre l'a priori (20 %) et les données (30 %).

### L'a posteriori au fil des visites

La cellule suivante trace l'a priori, puis l'a posteriori après 10, 50 et 500 visites, et affiche les trois estimations à chaque étape.

In [ ]:
grille = np.linspace(0, 1, 1001)
fig, ax = plt.subplots(figsize=(8, 4))
for N in [0, 10, 50, 500]:
  n = achats[:N].sum()
  loi = a_posteriori(alpha, beta, n, N)
  densite = loi.pdf(grille)
  nom = "A priori" if N == 0 else f"Après {N} visites ({n} achats)"
  ax.plot(grille, densite, label=nom)
  if N > 0:
    print(
      f"{N} visites : MLE {n / N:.1%}, MAP {grille[np.argmax(densite)]:.1%}, "
      f"moyenne a posteriori {loi.mean():.1%}"
    )
ax.axvline(0.12, color="black", linestyle="--", label="Vrai taux (12 %)")
ax.set_xlim(0, 0.5)
ax.set_xlabel("Taux de conversion")
ax.set_ylabel("Densité")
ax.legend()
plt.show()

Chaque courbe dit où le taux se trouve probablement : plus elle est étroite, plus la croyance est sûre. Avec peu de visites, la courbe est large et l'a priori pèse autant que les données ; avec 500, elle se resserre autour du vrai taux, et les trois estimations se rejoignent : quand les données abondent, l'a priori ne compte presque plus.

## 2. Reconnaître un vin : un classifieur bayésien naïf

Le jeu *Wine*, fourni par scikit-learn, décrit 178 vins d'une même région d'Italie, issus de trois cépages (numérotés 0, 1 et 2) : 13 mesures chimiques par vin (degré d'alcool, acidité, intensité de la couleur…), aux noms de colonnes en anglais. Le but : retrouver le cépage d'un vin d'après ses mesures. 30 % des vins sont mis de côté pour le test.

In [ ]:
vins = load_wine(as_frame=True)
X, y = vins.data, vins.target
X_entrainement, X_test, y_entrainement, y_test = train_test_split(
  X, y, test_size=0.3, stratify=y, random_state=0
)
print(f"{len(X_entrainement)} vins d'entraînement, {len(X_test)} de test")
X.head()

Le classifieur bayésien naïf choisit la classe la plus probable sachant les mesures, en les supposant indépendantes sachant la classe. Le cours estime ses probabilités par comptage ; pour des mesures continues comme celles-ci, `GaussianNB` décrit chaque mesure de chaque classe par une courbe en cloche (une loi normale), dont il apprend la moyenne et la variance.

*Entraînez un `GaussianNB` sur l'entraînement et affichez son exactitude sur le test. Affichez ensuite, avec `predict_proba`, la probabilité qu'il donne à chaque cépage pour les cinq premiers vins du test. Le modèle hésite-t-il ?*

**Documentation utile**

- [`sklearn.naive_bayes.GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
- [`GaussianNB.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB.predict_proba)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `GaussianNB()` n'a pas de paramètre à régler : `.fit(X_entrainement, y_entrainement)`, puis `.score(X_test, y_test)`, qui donne l'exactitude (la part de bonnes réponses).
- `modele.predict_proba(X_test.head())` renvoie un tableau d'une ligne par vin et d'une colonne par cépage, chaque ligne de somme 1 ; dans un `pd.DataFrame` arrondi, il se lit mieux.
- Exactitude attendue : 96,3 %.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = GaussianNB()
modele.fit(..., ...)
print(f"Exactitude : {modele.score(..., ...):.1%}")
probas = modele.predict_proba(...)
pd.DataFrame(probas, columns=["Cépage 0", "Cépage 1", "Cépage 2"]).round(3)
```

### Solution

In [ ]:
modele = GaussianNB()
modele.fit(X_entrainement, y_entrainement)
print(f"Exactitude : {modele.score(X_test, y_test):.1%}")
probas = modele.predict_proba(X_test.head())
pd.DataFrame(probas, columns=["Cépage 0", "Cépage 1", "Cépage 2"]).round(3)

96,3 % de bonnes réponses, avec un modèle qui s'entraîne en un instant. Mais il n'hésite presque jamais : ses probabilités sont proches de 0 ou de 1, même quand il se trompe (l'un des deux vins mal classés du test reçoit 91,5 % pour le mauvais cépage).

### Ce que le modèle a appris

Pour chaque cépage, l'histogramme du degré d'alcool des vins d'entraînement, et la courbe en cloche que le modèle en a tirée.

In [ ]:
mesure = "alcohol"
j = list(X.columns).index(mesure)
x = np.linspace(X[mesure].min(), X[mesure].max(), 200)
fig, ax = plt.subplots(figsize=(8, 4))
for cepage in modele.classes_:
  valeurs = X_entrainement.loc[y_entrainement == cepage, mesure]
  ax.hist(valeurs, bins=10, density=True, alpha=0.3, color=f"C{cepage}")
  courbe = stats.norm(modele.theta_[cepage, j], np.sqrt(modele.var_[cepage, j]))
  ax.plot(x, courbe.pdf(x), color=f"C{cepage}", label=f"Cépage {cepage}")
ax.set_xlabel("Degré d'alcool (% vol.)")
ax.set_ylabel("Densité")
ax.legend()
plt.show()

Tout le modèle tient en 3 × 13 courbes comme celles-ci : une par cépage et par mesure. Pour classer un vin, il multiplie les hauteurs de ses 13 courbes, comme si les mesures étaient indépendantes. Or plusieurs mesures disent la même chose (les phénols et les flavonoïdes vont ensemble) : le modèle compte plusieurs fois le même indice, d'où des probabilités trop tranchées. Il classe bien, mais ses probabilités ne sont pas à prendre au pied de la lettre.

## 3. La pluie, l'arroseur et l'herbe mouillée

Le réseau bayésien du cours : la pluie $R$ influence l'arroseur $A$ (on l'allume rarement quand il pleut), et tous deux mouillent l'herbe $H$. Chaque variable vaut 1 (vrai) ou 0 (faux). La cellule suivante reprend les tables de la figure du cours et en tire la loi jointe, le produit des tables : $P(R, A, H) = P(R)\,P(A \mid R)\,P(H \mid A, R)$, une ligne par combinaison des trois variables.

In [ ]:
# Tables du cours : P(R = 1), P(A = 1 | R) et P(H = 1 | A, R)
p_pluie = 0.2
p_arroseur = {0: 0.4, 1: 0.01}  # selon R
p_herbe = {(0, 0): 0.0, (0, 1): 0.8, (1, 0): 0.9, (1, 1): 0.99}  # selon (A, R)


# P(X = valeur) pour une variable qui vaut 1 avec la probabilité p
def bernoulli(p, valeur):
  return p if valeur == 1 else 1 - p


lignes = []
for R, A, H in itertools.product([1, 0], repeat=3):
  p = bernoulli(p_pluie, R) * bernoulli(p_arroseur[R], A) * bernoulli(p_herbe[A, R], H)
  lignes.append({"R": R, "A": A, "H": H, "P": p})
jointe = pd.DataFrame(lignes)
jointe

Inférer, c'est calculer la probabilité d'une variable sachant ce qu'on observe. Par énumération : on garde les lignes de la jointe qui s'accordent avec les observations, et on divise la probabilité de celles où la variable cherchée vaut 1 par leur total.

*Écrivez la fonction `inferer(jointe, cible, observations)`, qui renvoie $P(\text{cible} = 1 \mid \text{observations})$, `observations` étant un dictionnaire comme `{"H": 1}`. La cellule qui suit la solution lui pose trois questions.*

**Documentation utile**

- [Sélection par un masque de booléens dans pandas](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)

In [ ]:
def inferer(jointe, cible, observations):
  # Votre code ici
  ...

### Conseils supplémentaires

- Un masque de booléens, vrai pour chaque ligne au départ : `masque = pd.Series(True, index=jointe.index)`. Pour chaque `variable, valeur` de `observations.items()`, `masque &= jointe[variable] == valeur` ne garde vraies que les lignes qui s'accordent avec l'observation.
- `gardees = jointe[masque]` : les lignes gardées. Le numérateur est la somme de leur colonne `P` là où `cible` vaut 1, le dénominateur la somme de toute leur colonne `P`.
- Exemple à vérifier : sans observation, `inferer(jointe, "R", {})` vaut 0,2, l'a priori de la pluie.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def inferer(jointe, cible, observations):
  masque = pd.Series(True, index=jointe.index)
  for variable, valeur in observations.items():
    masque &= ...
  gardees = jointe[masque]
  return gardees.loc[..., "P"].sum() / gardees["P"].sum()
```

### Solution

In [ ]:
def inferer(jointe, cible, observations):
  masque = pd.Series(True, index=jointe.index)
  for variable, valeur in observations.items():
    masque &= jointe[variable] == valeur
  gardees = jointe[masque]
  return gardees.loc[gardees[cible] == 1, "P"].sum() / gardees["P"].sum()

### Trois questions au réseau

Il a plu avec quelle probabilité, sans rien savoir, puis en voyant l'herbe mouillée, puis en apprenant en plus que l'arroseur était allumé ?

In [ ]:
questions = {
  "P(R = 1)": {},
  "P(R = 1 | H = 1)": {"H": 1},
  "P(R = 1 | H = 1, A = 1)": {"H": 1, "A": 1},
}
for question, observations in questions.items():
  print(f"{question} = {inferer(jointe, 'R', observations):.3f}")

L'herbe mouillée fait passer la pluie de 20 % à 35,8 % : on remonte d'un effet à l'une de ses causes possibles, comme un diagnostic. Apprendre que l'arroseur était allumé la fait retomber à 0,7 % : l'arroseur explique à lui seul l'herbe mouillée, et on l'allume rarement quand il pleut.

L'énumération parcourt toute la jointe, $2^n$ lignes pour $n$ variables binaires : au-delà de quelques dizaines de variables, les bibliothèques de réseaux bayésiens (`pgmpy`, par exemple) calculent sans construire la jointe, en suivant la structure du graphe, ou approchent le résultat par échantillonnage.

## 4. La météo cachée : l'algorithme de Viterbi

Le modèle de Markov caché du cours : chaque jour, la pression est haute ($H$) ou basse ($L$), sans qu'on la voie ; on n'observe que le temps qu'il fait, Soleil, Nuage ou Pluie. Ses paramètres sont ceux de la figure du cours : les probabilités de départ $\pi$, les transitions $T$ (ligne : l'état de la veille, colonne : celui du jour) et les émissions $E$ (ligne : l'état, colonne : l'observation).

In [ ]:
etats = ["H", "L"]  # haute et basse pression
observations = ["Soleil", "Nuage", "Pluie"]
pi = np.array([0.7, 0.3])
T = np.array([[0.3, 0.7], [0.4, 0.6]])
E = np.array([[0.6, 0.1, 0.3], [0.2, 0.4, 0.4]])

# Cinq jours observés, chacun codé par sa position dans `observations`, en colonne
jours = ["Soleil", "Nuage", "Nuage", "Pluie", "Soleil"]
sequence = np.array([[observations.index(jour)] for jour in jours])
sequence.ravel()

*Créez un `CategoricalHMM` à deux états et donnez-lui ces paramètres (ses attributs `startprob_`, `transmat_` et `emissionprob_`). Trouvez ensuite, avec sa méthode `decode`, la suite d'états cachés la plus probable pour ces cinq jours, et affichez-la avec les noms des états. Est-ce celle du cours ?*

**Documentation utile**

- [`hmmlearn.hmm.CategoricalHMM`](https://hmmlearn.readthedocs.io/en/latest/api.html#hmmlearn.hmm.CategoricalHMM)
- [`CategoricalHMM.decode`](https://hmmlearn.readthedocs.io/en/latest/api.html#hmmlearn.hmm.CategoricalHMM.decode)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `CategoricalHMM(n_components=2)` crée un modèle à deux états. Ses paramètres s'affectent directement (`modele_meteo.startprob_ = pi`, de même `transmat_` et `emissionprob_`) : rien à entraîner, ils sont connus.
- `modele_meteo.decode(sequence, algorithm="viterbi")` renvoie deux valeurs : le logarithme de la probabilité du meilleur chemin, et ce chemin, un tableau d'indices d'états (0 pour H, 1 pour L).
- `[etats[i] for i in chemin]` donne les noms des états ; `np.exp` redonne la probabilité à partir de son logarithme.
- Attendu : le chemin (H, L, L, L, H), de probabilité 0,0016, la valeur de son dernier nœud dans le treillis de Viterbi du cours.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele_meteo = CategoricalHMM(n_components=2)
modele_meteo.startprob_ = pi
modele_meteo.transmat_ = ...
modele_meteo.emissionprob_ = ...
log_proba, chemin = modele_meteo.decode(..., algorithm="viterbi")
print("États les plus probables :", [etats[i] for i in ...])
print(f"Probabilité du chemin : {np.exp(...):.4f}")
```

### Solution

In [ ]:
modele_meteo = CategoricalHMM(n_components=2)
modele_meteo.startprob_ = pi
modele_meteo.transmat_ = T
modele_meteo.emissionprob_ = E
log_proba, chemin = modele_meteo.decode(sequence, algorithm="viterbi")
print("États les plus probables :", [etats[i] for i in chemin])
print(f"Probabilité du chemin : {np.exp(log_proba):.4f}")

### Probabilité de chaque état, jour par jour

Viterbi donne le meilleur chemin d'ensemble. La méthode `predict_proba` (l'algorithme Forward-Backward du cours) donne autre chose : pour chaque jour, la probabilité de chaque état sachant toute la séquence observée.

In [ ]:
tableau = pd.DataFrame(modele_meteo.predict_proba(sequence), columns=etats).round(3)
tableau.insert(0, "Observation", jours)
tableau

Ici, l'état le plus probable de chaque jour suit le chemin de Viterbi, ce qui n'est pas toujours le cas : le meilleur chemin d'ensemble n'est pas forcément fait des meilleurs états pris un à un. Ces probabilités disent en plus la confiance du modèle : 88,5 % de haute pression le premier jour, 70 % seulement de basse pression le quatrième.

## 5. Facultatif : une température et son incertitude

*Exercice facultatif, pour qui a fini les autres.*

Un capteur relève la température d'une serre au fil d'une journée, mais tombe en panne de 6 h à 17 h. Un processus gaussien prédit la température à toute heure avec son incertitude : une moyenne et une bande autour d'elle. Les mesures sont simulées.

In [ ]:
heures = np.array([0.5, 2, 4, 6, 17, 19, 21.5, 23.5])
bruit = np.random.default_rng(0).normal(0, 0.5, heures.size)
temperatures = 14 + 6 * np.sin((heures - 9) * np.pi / 12) + bruit
pd.DataFrame({"Heure": heures, "Température (°C)": temperatures.round(1)})

*Entraînez un `GaussianProcessRegressor` sur ces mesures, prédisez la température de 0 h à 24 h avec sa moyenne et son écart-type, puis tracez la moyenne, la bande « moyenne ± 2 écarts-types » et les mesures. Où la bande est-elle la plus large, et pourquoi ?*

**Documentation utile**

- [`sklearn.gaussian_process.GaussianProcessRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html)
- [`sklearn.gaussian_process.kernels.RBF`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.kernels.RBF.html)
- [`sklearn.gaussian_process.kernels.WhiteKernel`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.kernels.WhiteKernel.html)
- [`matplotlib.axes.Axes.fill_between`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Le noyau dit ce que le modèle suppose de la courbe : `RBF(length_scale=3)`, que deux heures proches (à environ 3 h près, une échelle que l'entraînement ajuste) ont des températures proches ; `+ WhiteKernel(noise_level=0.25)`, que chaque mesure a un peu de bruit.
- `GaussianProcessRegressor(kernel=noyau, normalize_y=True, random_state=0)` : `normalize_y=True` centre les températures avant l'entraînement.
- scikit-learn veut les entrées en colonne : `heures.reshape(-1, 1)` pour `.fit`, de même pour la grille de prédiction, `np.linspace(0, 24, 241)`.
- `.predict(..., return_std=True)` renvoie la moyenne et l'écart-type en chaque heure de la grille ; `ax.fill_between(x, bas, haut, alpha=0.3)` colore la bande entre `bas` et `haut`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
noyau = RBF(length_scale=3) + WhiteKernel(noise_level=0.25)
processus = GaussianProcessRegressor(kernel=noyau, normalize_y=True, random_state=0)
processus.fit(..., temperatures)
grille_heures = np.linspace(0, 24, 241)
moyenne, ecart_type = processus.predict(..., return_std=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grille_heures, moyenne, label="Moyenne a posteriori")
ax.fill_between(grille_heures, ..., ..., alpha=0.3, label="± 2 écarts-types")
ax.scatter(heures, temperatures, color="black", zorder=3, label="Mesures")
ax.set_xlabel("Heure")
ax.set_ylabel("Température (°C)")
ax.legend()
plt.show()
```

### Solution

In [ ]:
noyau = RBF(length_scale=3) + WhiteKernel(noise_level=0.25)
processus = GaussianProcessRegressor(kernel=noyau, normalize_y=True, random_state=0)
processus.fit(heures.reshape(-1, 1), temperatures)
grille_heures = np.linspace(0, 24, 241)
moyenne, ecart_type = processus.predict(grille_heures.reshape(-1, 1), return_std=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grille_heures, moyenne, label="Moyenne a posteriori")
ax.fill_between(
  grille_heures,
  moyenne - 2 * ecart_type,
  moyenne + 2 * ecart_type,
  alpha=0.3,
  label="± 2 écarts-types",
)
ax.scatter(heures, temperatures, color="black", zorder=3, label="Mesures")
ax.set_xlabel("Heure")
ax.set_ylabel("Température (°C)")
ax.legend()
plt.show()

La bande est étroite près des mesures et s'élargit pendant la panne, de 6 h à 17 h : loin de toute mesure, le modèle sait qu'il ne sait pas. Une régression ordinaire tracerait la même courbe sans dire où s'en méfier. C'est ce qu'apporte l'approche bayésienne : une prédiction accompagnée de son incertitude.

## 6. Facultatif : trois variantes d'une page, ε-glouton contre Thompson

*Exercice facultatif, pour qui a fini les autres.*

La boutique de la section 1 hésite maintenant entre trois versions de sa page produit, A, B et C. Chaque visite est montrée à l'une d'elles et finit par un achat (récompense 1) ou non (0) : c'est un bandit à trois bras. Montrer longtemps les trois versions à parts égales coûte des ventes ; ne plus montrer que celle qui semble la meilleure risque de garder la mauvaise. Deux stratégies arbitrent entre explorer et exploiter :

- **ε-glouton** (ε = 0,1) : une visite sur dix, une version au hasard ; les autres, la version au meilleur taux observé jusque-là ;
- **l'échantillonnage de Thompson** : chaque version a son a posteriori Bêta, mis à jour comme en section 1 ; à chaque visite, on tire un taux dans chacun et on montre la version au plus grand tirage.

Leur coût se mesure par le **regret** : après $T$ visites, $R_T = T\mu^\star - \sum_{t=1}^{T} \mu_{k_t}$, où $\mu_{k_t}$ est le vrai taux de la version montrée à la visite $t$ et $\mu^\star$ celui de la meilleure. C'est le nombre d'achats perdus, en moyenne, faute d'avoir toujours montré la meilleure version.

Les données sont simulées : les vrais taux, que la boutique ne connaît pas, sont de 10 %, 12 % et 15 %, assez proches pour que les premières visites ne tranchent pas. Pour que la comparaison ne tienne pas à un tirage chanceux, elle se répète 20 fois sur 10 000 visites : `resultats[r, t, k]` dit si la visite `t` de la répétition `r` achèterait devant la version `k`, et une stratégie ne lit que la case de la version qu'elle montre.

In [ ]:
# Les vrais taux de conversion des versions A, B et C, que la boutique ne connaît pas
taux_vrais = np.array([0.10, 0.12, 0.15])
nb_repetitions, nb_visites = 20, 10_000
# resultats[r, t, k] : la visite t de la répétition r achèterait-elle devant k ?
forme = (nb_repetitions, nb_visites, 3)
resultats = np.random.default_rng(131).random(forme) < taux_vrais
print(f"{nb_repetitions} répétitions de {nb_visites} visites, 3 versions")

*Écrivez les deux stratégies, `epsilon_glouton(resultats, epsilon, rng)` et `thompson(resultats, rng)`. Chacune reçoit les visites d'une répétition (`resultats[r]`, une ligne par visite, une colonne par version), choisit une version à chaque visite, n'apprend que la réponse de la visite à cette version (achat ou non) et renvoie le tableau des versions montrées. L'a priori de Thompson est l'uniforme Bêta(1, 1), celui du cours : la boutique n'a aucun historique de ces trois versions, et un même a priori pour toutes n'en favorise aucune.*

*Écrivez ensuite `regret_cumule(choix)`, qui renvoie le regret cumulé de chaque répétition visite après visite, faites jouer les deux stratégies (ε = 0,1) sur les 20 répétitions et tracez leur regret cumulé moyen sur un même graphique. Laquelle regrette le moins, et pourquoi ?*

**Documentation utile**

- [`numpy.random.Generator.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html)
- [`numpy.random.Generator.integers`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.integers.html)
- [`numpy.random.Generator.beta`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.beta.html)
- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)
- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)

In [ ]:
def epsilon_glouton(resultats, epsilon, rng):
  # Votre code ici
  ...


def thompson(resultats, rng):
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

- Une seule graine pour toute l'expérience : `rng = np.random.default_rng(1)`, passé aux deux stratégies. `rng.random() < epsilon` est vrai une fois sur dix pour `epsilon=0.1`, et `rng.integers(nb_versions)` tire alors une version parmi 0, 1 et 2.
- Taux observés : `succes / np.maximum(essais, 1)`, où `np.maximum` évite de diviser par zéro pour une version encore jamais montrée (elle compte pour 0) ; `np.argmax` donne la version au meilleur taux.
- Thompson : avec l'a priori Bêta(1, 1), la règle de la section 1 donne l'a posteriori Bêta(1 + succès, 1 + échecs). `rng.beta(1 + succes, 1 + echecs)` tire d'un coup un taux dans chacun des trois (un tableau de paramètres donne un tirage par case). `a_posteriori(1, 1, ...)` donnerait la même loi, mais créer un objet scipy à chaque visite est des dizaines de fois plus lent sur 200 000 visites.
- Regret : `taux_vrais[choix]` donne le vrai taux de la version montrée à chaque visite (l'indexation par un tableau d'indices), et `taux_vrais.max() - taux_vrais[choix]` son écart au meilleur : 0,05 pour A, 0,03 pour B, 0 pour C. `np.cumsum(..., axis=1)` le cumule visite après visite, répétition par répétition ; `.mean(axis=0)` fait la moyenne des répétitions.
- Le calcul prend quelques secondes : 2 stratégies × 20 répétitions × 10 000 visites.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def epsilon_glouton(resultats, epsilon, rng):
  nb_visites, nb_versions = resultats.shape
  succes, essais = np.zeros(nb_versions), np.zeros(nb_versions)
  choix = np.zeros(nb_visites, dtype=int)
  for t in range(nb_visites):
    if rng.random() < ...:
      k = rng.integers(...)
    else:
      k = np.argmax(...)
    succes[k] += resultats[t, k]
    essais[k] += 1
    choix[t] = k
  return choix


def thompson(resultats, rng):
  nb_visites, nb_versions = resultats.shape
  succes, echecs = np.zeros(nb_versions), np.zeros(nb_versions)
  choix = np.zeros(nb_visites, dtype=int)
  for t in range(nb_visites):
    k = np.argmax(rng.beta(..., ...))
    succes[k] += ...
    echecs[k] += ...
    choix[t] = k
  return choix


def regret_cumule(choix):
  return np.cumsum(..., axis=1)


rng = np.random.default_rng(1)
choix_glouton = np.array([epsilon_glouton(r, 0.1, rng) for r in resultats])
choix_thompson = np.array([thompson(r, rng) for r in resultats])
meilleure = np.argmax(taux_vrais)
fig, ax = plt.subplots(figsize=(8, 4))
strategies = {"ε-glouton (ε = 0,1)": choix_glouton, "Thompson": choix_thompson}
for nom, choix in strategies.items():
  regret = regret_cumule(choix).mean(axis=0)
  ax.plot(np.arange(1, nb_visites + 1), regret, label=nom)
  print(f"{nom} : regret {regret[4999]:.1f} à 5000 visites, {regret[-1]:.1f} à 10000")
  part = np.mean(choix[:, -1000:] == meilleure)
  print(f"  meilleure version montrée à {part:.1%} des 1000 dernières visites")
ax.set_xlabel("Visites")
ax.set_ylabel("Regret cumulé moyen")
ax.legend()
plt.show()
```

### Solution

In [ ]:
def epsilon_glouton(resultats, epsilon, rng):
  nb_visites, nb_versions = resultats.shape
  succes, essais = np.zeros(nb_versions), np.zeros(nb_versions)
  choix = np.zeros(nb_visites, dtype=int)
  for t in range(nb_visites):
    if rng.random() < epsilon:
      k = rng.integers(nb_versions)  # exploration : une version au hasard
    else:
      k = np.argmax(succes / np.maximum(essais, 1))  # le meilleur taux observé
    succes[k] += resultats[t, k]
    essais[k] += 1
    choix[t] = k
  return choix


def thompson(resultats, rng):
  nb_visites, nb_versions = resultats.shape
  succes, echecs = np.zeros(nb_versions), np.zeros(nb_versions)
  choix = np.zeros(nb_visites, dtype=int)
  for t in range(nb_visites):
    # Un taux tiré dans l'a posteriori Bêta(1 + succès, 1 + échecs) de chaque version
    k = np.argmax(rng.beta(1 + succes, 1 + echecs))
    succes[k] += resultats[t, k]
    echecs[k] += 1 - resultats[t, k]
    choix[t] = k
  return choix


def regret_cumule(choix):
  return np.cumsum(taux_vrais.max() - taux_vrais[choix], axis=1)


rng = np.random.default_rng(1)
choix_glouton = np.array([epsilon_glouton(r, 0.1, rng) for r in resultats])
choix_thompson = np.array([thompson(r, rng) for r in resultats])
meilleure = np.argmax(taux_vrais)
fig, ax = plt.subplots(figsize=(8, 4))
strategies = {"ε-glouton (ε = 0,1)": choix_glouton, "Thompson": choix_thompson}
for nom, choix in strategies.items():
  regret = regret_cumule(choix).mean(axis=0)
  ax.plot(np.arange(1, nb_visites + 1), regret, label=nom)
  print(f"{nom} : regret {regret[4999]:.1f} à 5000 visites, {regret[-1]:.1f} à 10000")
  part = np.mean(choix[:, -1000:] == meilleure)
  print(f"  meilleure version montrée à {part:.1%} des 1000 dernières visites")
ax.set_xlabel("Visites")
ax.set_ylabel("Regret cumulé moyen")
ax.legend()
plt.show()

Les deux courbes partent ensemble, puis se séparent : après 10 000 visites, ε-glouton a perdu en moyenne 82,7 achats, Thompson 52,2. Et l'écart se creuse : sur les 5 000 dernières visites, ε-glouton en perd encore 24,1, Thompson 7,5.

ε-glouton explore une visite sur dix quoi qu'il sache déjà : deux fois sur trois, il montre alors une version moins bonne, ce qui coûte au moins 2,7 achats par 1 000 visites, sans fin. Son regret croît en ligne droite, et il ne peut pas montrer la meilleure version à plus de 93,3 % des visites (90 % + 10 % / 3) : 88,8 % des 1 000 dernières. Le reste du temps, il se fie au meilleur taux observé, même sur peu de visites : dans l'une des 20 répétitions, il a montré B (12 %) à 9 029 visites sur 10 000, pour un regret de 294 à elle seule.

Thompson montre chaque version selon sa probabilité d'être la meilleure. À mesure que les a posteriori se resserrent, ceux de A et de B ne donnent presque plus jamais le plus grand tirage : l'exploration s'éteint d'elle-même (C reçoit 97,2 % des 1 000 dernières visites), et la courbe s'infléchit, vers le regret en $\ln T$ du cours.

## Conclusion

- **L'a posteriori combine l'a priori et les données** : avec peu de données, l'a priori retient les conclusions hâtives ; avec beaucoup, les données l'emportent, et le maximum de vraisemblance, le MAP et la moyenne a posteriori se rejoignent.
- **Le classifieur bayésien naïf** s'entraîne en un instant et classe souvent bien, mais ses probabilités sont trop tranchées : l'indépendance qu'il suppose est rarement vraie.
- **Un réseau bayésien** répond à toute question de la forme « cette cause, sachant ces effets » à partir de tables locales.
- **Un modèle de Markov caché** retrouve des états qu'on ne voit pas à partir d'observations : le meilleur chemin (Viterbi), et la confiance jour par jour (Forward-Backward).
- **Un processus gaussien** (facultatif) prédit une courbe et dit où il est incertain.
- **L'échantillonnage de Thompson** (facultatif) fait de l'a posteriori une stratégie : il montre chaque version selon sa probabilité d'être la meilleure, son exploration s'éteint à mesure que les données arrivent, et il regrette bien moins qu'ε-glouton, qui explore sans fin.